In [1]:
import yfinance as yf
from fredapi import Fred
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
import pandas_ta as ta
import itertools
from tqdm import tqdm

In [2]:
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.6
MIN_LIFT = 1.2
SELECTED_LABEL = "Trade_Profitable"
STOCK = "BTC-USD"
START_DATE = "2021-01-01"
END_DATE = "2025-04-01"

In [3]:
data = yf.download(STOCK, start=START_DATE, end=END_DATE).reset_index().rename(columns={"Date": "date"})
data.columns = data.columns.droplevel(1)
data.set_index("date", inplace=True)

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [4]:
def get_av_data(function, nm):
    fred = Fred(api_key="9ff6442c22eb1a25bb3ff361eceef633")
    data = fred.get_series(function, observation_start=add_months(START_DATE, -1), observation_end=add_months(END_DATE, 1))
    if len(data) > 0:   
        return pd.DataFrame(data.reset_index()).rename(columns={"index": "date", 0: nm}).fillna(method='ffill')
    else:
        return None

def add_months(date_str: str, months: int) -> str:
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")  # Convert string to datetime
    new_date = date_obj + relativedelta(months=months)  # Add or subtract months
    return new_date.strftime("%Y-%m-%d")  # Convert back to string

def categorize_rsi(rsi_series):
    conditions = [
        rsi_series < 30,
        rsi_series > 70,
    ]
    choices = ['bullish', 'bearish']
    return np.select(conditions, choices, default='neutral')

def calculate_barriers(price, volatility, Fu, Fl):
    upper = price + price * volatility * Fu
    lower = price - price * volatility * Fl
    return upper, lower

def assign_labels(df, Fu, Fl, Vt):
    labels = []
    for i in range(len(df) - Vt):
        window = df.iloc[i:i+Vt+1]
        price_start = window['Close'].iloc[0]
        vol = window['Volatility'].iloc[0]
        upper, lower = calculate_barriers(price_start, vol, Fu, Fl)
        price_max = window['Close'].max()
        price_min = window['Close'].min()
        if price_max >= upper:
            labels.append("bullish")
        elif price_min <= lower:
            labels.append("bearish")
        else:
            labels.append('neutral')
    labels.extend([None] * Vt)
    return labels

In [5]:
sentiments = pd.read_csv("cryptobert_btc_all_sentiments.csv")[["date", "positive", "neutral", "negative"]]
sentiments = sentiments.rename(columns={"positive": "pos_avg", "negative": "neg_avg"})
sentiments["compound_avg"] = sentiments["pos_avg"] - sentiments["neg_avg"]
sentiments['date'] = pd.to_datetime(sentiments['date'])
datasets = [sentiments]
data = data.sort_values("date")
for dataset in datasets:
    data = pd.merge_asof(
        data.sort_values("date"),
        dataset.sort_values("date"),
        on="date",
        direction="backward"
    )

In [6]:
rsi_windows = [7, 10, 14, 21]
volatility_windows = [5, 15, 30, 45]
roc_windows = [5, 8, 10]
tbl_windows = [8, 10, 12, 15]

# Track best result
best_lift = 0
best_params = {}
best_rules = None

# Grid search
for rsi_w, vol_w, roc_w, tbl_w in tqdm(itertools.product(rsi_windows, volatility_windows, roc_windows, tbl_windows)):

    temp = data.copy()
    
    temp["SEN_Above_Rolling"] = (temp["compound_avg"] > temp["compound_avg"].rolling(rsi_w).mean()).astype(int)
    temp["POS_Above_Rolling"] = (temp["pos_avg"] > temp["pos_avg"].rolling(rsi_w).mean()).astype(int)
    temp["NEG_Above_Rolling"] = (temp["neg_avg"] > temp["neg_avg"].rolling(rsi_w).mean()).astype(int)

    temp["RSI"] = ta.rsi(temp["Close"], length=rsi_w)
    temp['RSI_Label'] = categorize_rsi(temp['RSI'])
    rsi_dummies = pd.get_dummies(temp['RSI_Label'], prefix='RSI')
    temp = pd.concat([temp, rsi_dummies], axis=1)
    
    temp["ROC"] = temp["Close"].pct_change(periods=roc_w) * 100
    roc_std = temp['ROC'].rolling(window=roc_w).std()
    temp['ROC_Label'] = np.where(temp['ROC'] > roc_std, 'bullish', np.where(temp['ROC'] < -roc_std, 'bearish', 'neutral'))
    roc_dummies = pd.get_dummies(temp['ROC_Label'], prefix='ROC')
    temp = pd.concat([temp, roc_dummies], axis=1)
    
    temp['LogReturn'] = np.log(temp['Close'] / temp['Close'].shift(1))
    temp['Volatility'] = temp['LogReturn'].ewm(span=vol_w).std()
    vol_threshold = temp['Volatility'].rolling(window=vol_w).quantile(0.7)
    temp['Volatility_High'] = (temp['Volatility'] > vol_threshold).astype(int)
    temp['Volatility_Low'] = (temp['Volatility'] < vol_threshold).astype(int)
    
    temp["TBL_Label"] = assign_labels(temp, Fu=1.5, Fl=2.5, Vt=tbl_w)
    tbl_dummies = pd.get_dummies(temp["TBL_Label"], prefix="TBL")
    temp = pd.concat([temp, tbl_dummies], axis=1)
    
    temp["Trade_Profitable"] = (temp["Close"].shift(-1) > temp["Close"]).astype(int)
    
    temp = temp.dropna()
    # Keep only binary columns for Apriori
    binarized = temp[["TBL_bullish", "SEN_Above_Rolling", "POS_Above_Rolling", "NEG_Above_Rolling", "RSI_bearish", "RSI_bullish", "RSI_neutral", "ROC_bearish", "ROC_bullish", "ROC_neutral", "Volatility_High", "Volatility_Low"]].astype(bool)

    # Apriori
    freq_sets = apriori(binarized, min_support=MIN_SUPPORT, use_colnames=True)
    rules = association_rules(freq_sets, metric="confidence", min_threshold=MIN_CONFIDENCE)

    # Select rules predicting Trade_Profitable with good lift
    match = rules[
        (rules['consequents'] == {'TBL_bullish'}) &
        (rules['lift'] > MIN_LIFT) &
        (rules['lift'] > best_lift)
    ]

    if not match.empty:
        best_lift = match['lift'].max()
        best_params = {
            'RSI_window': rsi_w,
            'VOL_window': vol_w,
            'ROC_window': roc_w,
            'TBL_window': tbl_w
        }
        best_rules = match.sort_values(by='lift', ascending=False).head(3)

print("Best configuration:", best_params)
print("Best lift:", best_lift)
print("Top rule:")
print(best_rules[['antecedents', 'support', 'confidence', 'lift']])

192it [01:12,  2.66it/s]

Best configuration: {'RSI_window': 14, 'VOL_window': 45, 'ROC_window': 5, 'TBL_window': 10}
Best lift: 2.068313953488372
Top rule:
                                           antecedents   support  confidence  \
184   (Volatility_Low, RSI_bullish, SEN_Above_Rolling)  0.011947         1.0   
204   (POS_Above_Rolling, RSI_bullish, Volatility_Low)  0.011947         1.0   
423  (Volatility_Low, POS_Above_Rolling, RSI_bullis...  0.010541         1.0   

         lift  
184  2.068314  
204  2.068314  
423  2.068314  
